## Tips

1. If you want to run the `DeSCOPE_LOO` model, you need to run the code in both `1. DeSCOPE` and `2. DeSCOPE_LOO`.  

   - `1. DeSCOPE` is used to tokenize the cell types for fine-tuning.  

   - `2. DeSCOPE_LOO` is used to tokenize the cell types for pretraining.  

2. When tokenizing the pretraining cell types, make sure that ***the names of the control cells are consistent across all h5ad files***.  

3. Only set `skip_raw_counts_check` to `True` if you are certain that your h5ad files contain raw counts.

4. If your .h5ad file contains multi-gene knockouts, ensure the gene symbols are concatenated with a plus sign (+) (e.g., **'GeneA+GeneB'**).

In [ ]:
from descope.utils import DuplicatedFeatureHandling, load_gene_names_engine
from descope.tokenizer import TokenizerForRNA, tokenize_adata_to_hf_dataset_for_rna

### 1. DeSCOPE

In [ ]:
cellline = ["H1", "K562_ESSENTIAL", "JURKAT", "RPE1", "HEPG2"]  # all cell lines

for c in cellline:
    tokenize_adata_to_hf_dataset_for_rna(
        adata=f"/fse/home/wupengpeng/perturbation_data_origin/RNA/{c}/{c}.h5ad",
        cell_line_name=c,
        perts_to_exclude=load_gene_names_engine(f"/fse/home/wupengpeng/perturbation_data_origin/RNA/{c}/Test_Info_{c}.csv"),
        target_sum=1e4 if c != "H1" else 5e4,
        pert_col="gene" if c != "H1" else "target_gene",
        ctrl_name="non-targeting",
        skip_raw_counts_check=True,
        save_dir=f"/fse/home/wupengpeng/perturbation_data_origin/RNA/{c}/tokenized_dataset"
    )

### 2. DeSCOPE_LOO

In [ ]:
cellline = ["H1", "K562_ESSENTIAL", "JURKAT", "RPE1", "HEPG2"]  # all cell lines

for cellline_ft in cellline:
    # cellline_ft: cell line for finetune
    cellline_pt = [c for c in cellline if c != cellline_ft]  # cell line for pretrain

    tokenizer = TokenizerForRNA(
        cell_line_ft=f"/fse/home/wupengpeng/perturbation_data_origin/RNA/{cellline_ft}/{cellline_ft}.h5ad",
        target_sum=1e4 if cellline_ft != "H1" else 5e4,
        pert_col="gene" if cellline_ft != "H1" else "target_gene",
        gene_names_col="gene_name" if cellline_ft != "H1" else None
    )

    tokenizer.tokenize(
        # NOTE: Please ensure that the names of control cells in cell_line_pt.obs[pert_col] are consistent across all datasets.
        cell_line_pt=[f"/fse/home/wupengpeng/perturbation_data_origin/RNA/{cellline}/{cellline}.h5ad" for cellline in cellline_pt],
        cell_line_name=cellline_pt,
        pert_col=["gene" if cellline != "H1" else "target_gene" for cellline in cellline_pt],
        gene_names_col=["gene_name" if cellline != "H1" else None for cellline in cellline_pt],
        save_dir=f"/fse/home/wupengpeng/perturbation_data_origin/RNA/tokenized_dataset/loo_{cellline_ft}",
        apply_pert_gene_filter=False,
        duplicated_features_handling=DuplicatedFeatureHandling.max_pooling,
        skip_raw_counts_check=True  # NOTE: If you are certain that your input data consists of raw counts, you can skip this check to save time.
    )